# DGD Test Inference (Algorithm 2)

Loads the frozen `best/` decoder + GMM and optimizes a fresh representation layer for the genuinely held-out test split -- the first real use of `test_loader` anywhere in this codebase. See `dgd_generative_model.ipynb`, "Algorithm 2 -- Inference", for the formal definition this notebook implements.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn.functional as F

from omegaconf import OmegaConf
from hydra import initialize, compose
from tgmm import ClusteringMetrics

current_dir = Path.cwd()
if 'notebooks' in current_dir.parts:
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.data import create_dataloaders, collect_all_labels, collect_class_samples
from src.models import RepresentationLayer, ConvDecoder
from src.utils import setup_device, set_random_seed, setup_cuml_acceleration
from src.utils.checkpoint import load_checkpoint
from src.visualization import plot_latent_space, plot_images_by_class

device = setup_device(verbose=True)
set_random_seed(seed=42, device=device)
setup_cuml_acceleration(verbose=True)


In [ ]:
with initialize(version_base=None, config_path="../config"):
    config = compose(config_name="config")

config.data.root_dir = str(project_root / "data")
config.paths.models_dir = str(project_root / "models")
config.paths.figures_dir = str(project_root / "figures")

# Re-derive the same 3-way split independently (same random_seed as training),
# so `test_loader` here is exactly the held-out split carved out during training
# and never optimized against.
train_loader, val_loader, test_loader, class_names = create_dataloaders(config)
print(f"Test loader: {len(test_loader)} batches, {len(test_loader.dataset)} samples")


In [ ]:
def decoder_factory():
    return ConvDecoder(
        latent_dim=config.model.representation.n_features,
        hidden_dims=config.model.decoder.hidden_dims,
        output_channels=config.model.decoder.output_channels,
        output_size=config.model.decoder.output_size,
        activation=config.model.decoder.activation,
        final_activation=config.model.decoder.final_activation,
        dropout_rate=config.model.decoder.dropout_rate,
        init_size=config.model.decoder.init_size,
    )

best_dir = Path(config.paths.models_dir) / config.experiment_name / "best"
checkpoint = load_checkpoint(best_dir, decoder_factory, device=device)

decoder = checkpoint['decoder']
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

gmm = checkpoint['gmm']
if gmm is None:
    raise RuntimeError(
        f"No fitted GMM found in {best_dir} -- was training.first_epoch_gmm ever reached "
        "during training? Re-run dgd_training_demo.ipynb first."
    )

meta = checkpoint['metadata']
print(f"Loaded frozen decoder + GMM from {best_dir} "
      f"(best_epoch={meta.get('best_epoch')}, best_val_loss={meta.get('best_val_loss'):.4f})")


In [ ]:
# Algorithm 2, line 1: initialize a fresh representation layer for the new
# (held-out) data, using the same generic init distribution training used for
# Z_0 -- not a GMM-sample init, matching how the codebase already initializes
# val_rep alongside rep in DGDTrainer._create_model_components.
model_config = config.model
distribution = model_config.representation.distribution

if distribution == 'pca':
    print("Representation distribution is 'pca'; using 'normal' for the test layer "
          "instead (matches how val_rep is initialized under PCA during training).")
    test_distribution = 'normal'
    dist_params = {}
else:
    test_distribution = distribution
    dist_params = {}
    if hasattr(model_config.representation, 'radius'):
        dist_params['radius'] = model_config.representation.radius
    for param in ['mean', 'cov', 'low', 'high', 'loc', 'scale', 'scale_matrix',
                 'rate', 'df', 'mu', 'alpha', 'beta', 'delta']:
        if hasattr(model_config.representation, param):
            dist_params[param] = getattr(model_config.representation, param)

test_rep = RepresentationLayer(
    dim=model_config.representation.n_features,
    n_samples=len(test_loader.dataset),
    dist=test_distribution,
    dist_params=dist_params,
    device=device,
)
print(f"Initialized test representation layer: {test_rep.n_rep} samples x {test_rep.dim} dims")


In [ ]:
# Algorithm 2, lines 2-5: optimize test_rep alone against the frozen decoder+GMM.
# M0 (prior_warmup_steps) < M (epochs): reconstruction-only warm-up before the
# GMM prior term is added, so a fresh z isn't dominated by the prior gradient
# before it has any reconstruction signal to work with.
lr_config = config.training.lr_scheduler
rep_config = config.training.optimizer.representation

test_optimizer = torch.optim.AdamW(
    test_rep.parameters(),
    lr=rep_config.lr,
    betas=tuple(rep_config.betas),
    eps=rep_config.eps,
    weight_decay=rep_config.weight_decay,
    amsgrad=rep_config.get('amsgrad', False),
)

M = config.training.inference.epochs
M0 = config.training.inference.prior_warmup_steps
assert M0 < M, "training.inference.prior_warmup_steps must be < training.inference.epochs"

if lr_config.get('enabled', False):
    max_lr = lr_config.get('max_lr_representation', None) or rep_config.lr
    test_scheduler = torch.optim.lr_scheduler.OneCycleLR(
        test_optimizer,
        max_lr=max_lr,
        total_steps=M,
        pct_start=lr_config.get('pct_start', 0.3),
        anneal_strategy=lr_config.get('anneal_strategy', 'cos'),
        div_factor=lr_config.get('div_factor', 25.0),
        final_div_factor=lr_config.get('final_div_factor', 10000.0),
        cycle_momentum=lr_config.get('cycle_momentum', True),
        base_momentum=lr_config.get('base_momentum', 0.85),
        max_momentum=lr_config.get('max_momentum', 0.95),
        three_phase=lr_config.get('three_phase', False),
    )
else:
    test_scheduler = None

lambda_gmm = config.training.lambda_gmm
n_test = len(test_loader.dataset)
print(f"Optimizing {n_test} test representations for {M} steps (prior warm-up: {M0} steps)...")

for m in range(1, M + 1):
    test_optimizer.zero_grad()

    total_loss = 0.0
    total_recon = 0.0
    total_gmm = 0.0

    for index, x, _ in test_loader:
        x, index = x.to(device), index.to(device)

        z = test_rep(index)
        y = decoder(z)
        recon_loss = F.mse_loss(y, x, reduction='sum')

        if m >= M0:
            gmm_error = -lambda_gmm * torch.sum(gmm.score_samples(z))
            loss = recon_loss + gmm_error
        else:
            gmm_error = torch.tensor(0.0, device=device)
            loss = recon_loss

        loss.backward()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_gmm += gmm_error.item()

    test_optimizer.step()
    if test_scheduler is not None:
        test_scheduler.step()

    if m % max(1, M // 10) == 0 or m == M:
        print(f"Step {m}/{M}: loss={total_loss/n_test:.4f}, recon={total_recon/n_test:.4f}, gmm={total_gmm/n_test:.4f}")

print("Test representation optimization complete.")


In [ ]:
test_inference_dir = Path(config.paths.models_dir) / config.experiment_name / "test_inference"
test_inference_dir.mkdir(parents=True, exist_ok=True)
test_rep.save(str(test_inference_dir / "test_representation.pt"))
print(f"Saved optimized test representations to {test_inference_dir / 'test_representation.pt'}")


In [ ]:
figures_dir = Path(config.paths.figures_dir) / f"{config.experiment_name}_test_inference"
figures_dir.mkdir(parents=True, exist_ok=True)

test_labels = collect_all_labels(test_loader)

with torch.no_grad():
    predicted_labels = gmm.predict(test_rep.z.detach())

cluster_metrics = ClusteringMetrics()
test_ari = cluster_metrics.adjusted_rand_score(test_labels, predicted_labels)

max_silhouette_samples = 5000
reps_for_silhouette = test_rep.z.detach()
labels_for_silhouette = predicted_labels
if reps_for_silhouette.size(0) > max_silhouette_samples:
    idx = torch.randperm(reps_for_silhouette.size(0), device=reps_for_silhouette.device)[:max_silhouette_samples]
    reps_for_silhouette = reps_for_silhouette[idx]
    labels_for_silhouette = labels_for_silhouette[idx]
test_silhouette = cluster_metrics.silhouette_score(reps_for_silhouette, labels_for_silhouette, gmm.n_components)

print(f"Test ARI: {test_ari:.4f}, Test Silhouette: {test_silhouette:.4f}")

plot_latent_space(
    representations=test_rep.z.detach(), labels=test_labels, gmm=gmm, class_names=class_names,
    title=f"Test Latent Space (Algorithm 2 inference) - ARI: {test_ari:.4f}, Sil: {test_silhouette:.4f}",
    save_path=str(figures_dir / "latent_test.png"), show=False,
)

indices_test_sample, images_test_sample, labels_test_sample = collect_class_samples(
    test_loader, n_per_class=5, n_classes=len(class_names)
)
indices_test_sample = indices_test_sample.to(device)

with torch.no_grad():
    recon_test_sample = decoder(test_rep(indices_test_sample))

plot_images_by_class(
    images=recon_test_sample, labels=labels_test_sample, class_names=class_names,
    title="Test: Reconstructed Images by Class (Algorithm 2 inference)", n_per_class=5, cmap='viridis',
    save_path=str(figures_dir / "recon_test.png"), show=False,
)

print(f"Figures written to {figures_dir}")
